Hybrid Search langchain

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['api_key']= os.getenv("pinecone_api_key")

In [12]:
from langchain_community.retrievers import PineconeHybridSearchRetriever #can dp both semantic and keyword search
from pinecone import Pinecone, ServerlessSpec
index_name = "hybrid-search-langchain-pinecone"
#initialise pinecone client
pc = Pinecone(api_key=api_key)

#create the index 
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name = index_name,
        dimension=384, #dimension of dense vector
        metric='dotproduct', ## sparse values supported for dotproduct
        spec= ServerlessSpec(cloud='aws',region='us-east-1')

    )



In [13]:
index = pc.Index(index_name)
index

c:\Users\acer\Desktop\LANGCHAIN\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [23]:
## vector embedding and sparse matrix
hf_token = os.getenv("HF_TOKEN")

from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name= "all-MiniLM-L6-v2")
embeddings


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
from pinecone_text.sparse import BM25Encoder #uses TFIDF technique by default
bm25_encoder = BM25Encoder().default()
bm25_encoder #for sparse vector convertions

In [25]:
sentences = ['In 2023, I visited paris',
             "In 2022, I visited New York",
             "in 2021, In visited New orleans"]
#tfidf values on these sentences
bm25_encoder.fit(sentences)

#store the values to a json file
bm25_encoder.dump("bm25_values.json")


100%|██████████| 3/3 [00:00<00:00, 52.62it/s]


In [26]:
retriever = PineconeHybridSearchRetriever(embeddings=embeddings, sparse_encoder=bm25_encoder,index = index)


In [27]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000002582D2FE790>, index=<pinecone.db_data.index.Index object at 0x000002586EC23A10>)

In [29]:
retriever.add_texts(
    ['In 2023, I visited paris',
    "In 2022, I visited New York",
    "in 2021, In visited New orleans"]
)

100%|██████████| 1/1 [00:01<00:00,  1.50s/it]


In [32]:
retriever.invoke("What city did i visit first")

[Document(metadata={'score': 0.232817784}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.21249935}, page_content='In 2023, I visited paris'),
 Document(metadata={'score': 0.178239}, page_content='in 2021, In visited New orleans')]

In [ ]:
retriever.invoke("Out of these documents, which year was earliest?")

[Document(metadata={'score': 0.097814776}, page_content='In 2023, I visited paris'),
 Document(metadata={'score': 0.087659508}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.052238524}, page_content='in 2021, In visited New orleans')]

: 